# Задание 1. Теория

## 1. Что такое PySpark и в чем его главное отличие от Pandas?

**PySpark** — это Python API для фреймворка **Apache Spark**, предназначенного для распределенной обработки больших объемов данных.

PySpark позволяет писать код на Python, но вычисления выполняются движком Spark. Данные при этом могут быть распределены между несколькими процессами или несколькими узлами кластера.

Основной объект для табличной обработки данных — `DataFrame`.

Например:

```python
df = spark.read.csv("data.csv", header=True, inferSchema=True)
df.groupBy("category").count().show()
```

### Отличие PySpark от Pandas

Главное отличие заключается в **модели выполнения вычислений**.

**Pandas** обычно работает в рамках одного процесса и одной машины. DataFrame в основном хранится в оперативной памяти компьютера:

Если набор данных значительно превышает доступную оперативную память, работа Pandas становится затруднительной.

PySpark изначально рассчитан на распределенную обработку:

```text
                  Spark Driver
                       |
          +------------+------------+
          |            |            |
          v            v            v
      Executor 1   Executor 2   Executor 3
          |            |            |
       Part 1       Part 2       Part 3
```

Один DataFrame может быть разделен на **partitions**, которые параллельно обрабатываются разными Executor'ами.

Основные различия:

| Pandas | PySpark |
|---|---|
| Обычно работает на одной машине | Может работать на кластере |
| Данные в основном находятся в памяти одного процесса | Данные распределяются по partitions |
| Вычисления преимущественно выполняются сразу | Используется Lazy Evaluation |
| Подходит для небольших и средних данных | Предназначен в том числе для больших данных |
| Нет распределенного Driver/Executor выполнения | Используется архитектура Driver + Executors |

Таким образом, главное преимущество PySpark — возможность **масштабировать обработку данных на несколько вычислительных узлов**.

---

## 2. Что такое PySpark MLlib?

**MLlib** — библиотека машинного обучения Apache Spark.

В PySpark современные средства машинного обучения в основном доступны через пакет:

```python
pyspark.ml
```

MLlib предоставляет готовые инструменты для:

- классификации;
- регрессии;
- кластеризации;
- рекомендательных систем;
- обработки признаков;
- построения ML Pipeline;
- оценки качества моделей;
- подбора гиперпараметров.

Например:

```python
from pyspark.ml.classification import LogisticRegression

lr = LogisticRegression(
    featuresCol="features",
    labelCol="label"
)

model = lr.fit(train_df)

predictions = model.transform(test_df)
```

Особенность MLlib заключается в том, что алгоритмы могут работать с **распределенными Spark DataFrame**.

То есть вместо предварительного переноса всего датасета в память одной машины Spark может распределять обработку данных между Executor'ами.

Типичная схема MLlib:

```text
DataFrame
   |
   v
Подготовка признаков
   |
   v
ML Pipeline
   |
   v
Обучение модели
   |
   v
Model
   |
   v
Predictions
```

Для объединения этапов обработки данных используется `Pipeline`.

Например:

```text
StringIndexer
     |
     v
VectorAssembler
     |
     v
StandardScaler
     |
     v
LogisticRegression
```

Это позволяет описать подготовку данных и обучение модели как одну последовательность операций.

---

## 3. Как устроена базовая архитектура Spark-приложения: Driver и Executors?

Spark-приложение состоит из двух основных типов процессов:

1. **Driver**
2. **Executors**

### Driver

**Driver** — главный управляющий процесс Spark-приложения.

Он:

- запускает пользовательскую программу;
- создает `SparkSession`;
- анализирует операции над DataFrame;
- строит план выполнения;
- разбивает вычисления на задачи;
- распределяет задачи между Executor'ами;
- отслеживает выполнение задач;
- получает результаты вычислений.

### Executors

**Executor** — рабочий процесс Spark.

Executor:

- получает задачи от Driver;
- выполняет обработку partitions;
- хранит промежуточные данные;
- может хранить закэшированные данные;
- возвращает результаты Driver'у.


Если необходимо обработать DataFrame из 100 partitions, Spark может распределить эти partitions между несколькими Executor'ами.

---

## 4. В чем концепция Lazy Evaluation в PySpark?

Spark использует концепцию **Lazy Evaluation — ленивых вычислений**.

Это означает, что большинство операций над DataFrame **не выполняются сразу после вызова метода**.

Например:

```python
df2 = df.filter(df.age > 18)
df3 = df2.select("name", "age")
```

После выполнения этих строк Spark обычно еще не начал читать и обрабатывать весь набор данных.

Вместо этого он запоминает последовательность операций и формирует **логический план вычислений**.

Реальное выполнение начинается только после вызова операции, которой необходим результат, например:

```python
df3.show()
```

или:

```python
df3.count()
```

После этого Spark может оптимизировать сразу всю цепочку операций.

Например, вместо:

```text
прочитать все столбцы
       |
       v
выполнить filter
       |
       v
оставить два столбца
```

Spark может определить, что программе нужны только конкретные столбцы, и по возможности не читать остальные.

Таким образом:

```text
Transformations
      |
      v
Логический план
      |
      v
Optimization
      |
      v
Action
      |
      v
Фактическое выполнение
```

Главное преимущество Lazy Evaluation — возможность **оптимизировать полный план вычислений до начала его выполнения**.

---

## 5. В чем разница между Transformations и Actions?

Операции Spark можно условно разделить на две основные категории:

- **Transformations**
- **Actions**

### Transformations

Transformation создает новый DataFrame на основе существующего, но из-за Lazy Evaluation обычно не запускает вычисление сразу.

Примеры:

```python
df.filter(...)
df.select(...)
df.withColumn(...)
df.groupBy(...)
df.join(...)
df.orderBy(...)
```

### Actions

Action требует получить реальный результат и поэтому запускает вычисление подготовленного плана.

Примеры:

```python
df.show()
df.count()
df.collect()
df.first()
df.take(10)
```
 **Transformations описывают, что нужно сделать с данными, а Actions заставляют Spark реально выполнить эти операции**.

---

## 6. Что такое Shuffle?

**Shuffle** — это перераспределение данных между partitions, а часто и между разными Executor'ами.

Предположим, данные распределены следующим образом:

```text
Executor 1:
(A, 10)
(B, 20)

Executor 2:
(A, 30)
(C, 40)

Executor 3:
(B, 50)
(C, 60)
```

Требуется выполнить:

```python
df.groupBy("key").sum("value")
```

Для вычисления суммы все записи с одинаковым ключом должны оказаться вместе.

Spark должен перераспределить данные:

```text
До Shuffle:

Executor 1      Executor 2      Executor 3
A, B            A, C            B, C

             |
             | Shuffle
             v

После Shuffle:

Partition 1     Partition 2     Partition 3
A, A            B, B            C, C
```

После этого можно выполнить агрегацию:

```text
A -> 40
B -> 70
C -> 100
```

Shuffle часто возникает при:

- `groupBy`;
- `join`;
- `distinct`;
- `orderBy`;
- `repartition`;
- различных агрегациях.

Shuffle является дорогостоящей операцией, потому что может включать:

1. сериализацию данных;
2. запись промежуточных данных;
3. передачу данных по сети;
4. чтение данных другими Executor'ами;
5. сортировку и объединение данных.

Поэтому уменьшение количества ненужных Shuffle — один из важных способов оптимизации Spark-приложений.

---

## 7. Что такое Data Spill и почему его надо избегать?

Правильный термин — **Data Spill** или просто **Spill**.

Spill возникает, когда Spark не может разместить необходимые промежуточные данные в доступной оперативной памяти Executor'а и вынужден временно записывать их на диск.


### Почему Spill ухудшает производительность?

Оперативная память намного быстрее дискового ввода-вывода.

При Spill возникают дополнительные операции:

```text
данные
  |
  v
запись на диск
  |
  v
чтение с диска
  |
  v
дальнейшая обработка
```

Кроме того, Spill может увеличить:

- нагрузку на диск;
- время Shuffle;
- время сортировки;
- общее время выполнения Stage.

### Почему возникает Spill?

Основные причины:

- слишком большие partitions;
- большой объем данных при `groupBy`;
- тяжелые `join`;
- сортировки;
- недостаточный объем памяти Executor;
- неудачное количество partitions;
- сильный **data skew**, когда одна partition намного больше остальных.

Полностью исключить Spill не всегда возможно, но большое его количество обычно означает, что приложение следует оптимизировать.

---

## 8. Что такое Apache Parquet и зачем он нужен?

**Apache Parquet** — файловый формат для хранения структурированных данных, широко используемый в Spark и других системах обработки больших данных.

Главная особенность Parquet — **колоночное хранение данных**.

Предположим, имеется таблица:

| id | name | age |
|---:|---|---:|
| 1 | Anna | 21 |
| 2 | Ivan | 25 |
| 3 | Alex | 30 |

При условном построчном хранении данные расположены примерно так:

```text
1, Anna, 21
2, Ivan, 25
3, Alex, 30
```

То есть:

```text
Row 1 -> id, name, age
Row 2 -> id, name, age
Row 3 -> id, name, age
```

При колоночном хранении:

```text
id:
1, 2, 3

name:
Anna, Ivan, Alex

age:
21, 25, 30
```

То есть данные одного столбца физически хранятся рядом.

### Почему колоночное хранение полезно?

Предположим, таблица содержит 100 столбцов, но запросу нужны только два:

```python
df.select("age", "salary")
```

В построчном формате системе часто приходится читать данные строк, содержащих остальные столбцы.

Parquet позволяет Spark прочитать в основном только необходимые столбцы:

```text
100 columns
     |
     v
Запрос нужны только:
age
salary
     |
     v
Parquet читает нужные columns
```

Это называется **column pruning**.

В результате уменьшается количество данных, которые необходимо считать с диска.

### Хорошее сжатие

Колоночный формат также хорошо сжимается.

Например:

```text
country:
USA
USA
USA
USA
USA
USA
Germany
Germany
Germany
```

Так как соседние значения имеют одинаковый тип и часто повторяются, для них можно эффективно применять алгоритмы сжатия и специальные способы кодирования.

### Parquet хранит схему

Parquet хранит информацию о типах данных:

```text
id     -> integer
name   -> string
age    -> integer
salary -> double
```

Поэтому Spark обычно не требуется заново определять типы данных, как это часто происходит при чтении CSV.

### Predicate Pushdown

Parquet также позволяет использовать оптимизацию **Predicate Pushdown**.

Например:

```python
df.filter(df.age > 50)
```

Spark может использовать метаданные Parquet, чтобы не читать некоторые блоки файла, если известно, что подходящих значений там нет.

Например:

```text
Block 1: age 18-25   -> пропустить
Block 2: age 25-40   -> пропустить
Block 3: age 51-70   -> читать
```

### Parquet и CSV

Упрощенно:

| CSV | Parquet |
|---|---|
| Текстовый формат | Бинарный формат |
| Построчное представление | Колоночное хранение |
| Схема не является полноценной частью формата | Схема хранится в файле |
| Сжатие обычно менее эффективно | Хорошо сжимает колоночные данные |
| Часто приходится читать всю строку | Можно читать только необходимые столбцы |
| Удобен для просмотра человеком | Оптимизирован для аналитической обработки |

Поэтому Parquet особенно эффективен для Spark, где часто выполняются запросы только к части столбцов больших таблиц.

---

## Итоговая схема работы Spark

Общую модель можно представить следующим образом:

```text
                PySpark-программа
                       |
                       v
                     Driver
                       |
                строит план
                       |
             Lazy Evaluation
                       |
             +---------+---------+
             |                   |
             v                   v
      Transformation       Transformation
             |                   |
             +---------+---------+
                       |
                     Action
                       |
                       v
                   Spark Job
                       |
              +--------+--------+
              |        |        |
              v        v        v
           Executor Executor Executor
              |        |        |
              v        v        v
         Partition Partition Partition
              |
              +---- Shuffle ----+
              |
              v
       промежуточные данные
              |
       RAM, при нехватке
              |
              v
            Spill
              |
              v
             Disk
```

Основная идея Spark состоит в том, чтобы представить обработку данных как набор распределенных операций, оптимизировать их до выполнения и затем параллельно выполнить на Executor'ах.

In [1]:
import os
import sys

os.environ["JAVA_HOME"] = "/opt/homebrew/opt/openjdk@21"

import pyspark 
from pyspark.sql import SparkSession 
from pyspark.sql.functions import col, avg, count, split, explode, lit
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, FloatType, LongType

spark = (SparkSession.builder 
    .appName("MovieLens Lab") 
    .master("local[*]") 
    .getOrCreate() 
)

spark.sparkContext.setLogLevel("ERROR")

movies_schema = StructType([
    StructField("movie_id", IntegerType(), nullable=False),
    StructField("title", StringType(), nullable=True),
    StructField("genres", StringType(), nullable=True)
])

ratings_schema = StructType([
    StructField("user_id", IntegerType(), nullable=False),
    StructField("movie_id", IntegerType(), nullable=False),
    StructField("rating", IntegerType(), nullable=True),
    StructField("timestamp", LongType(), nullable=True) 
])

base_path = os.path.abspath("ml-1m")
movies_path = f"file://{base_path}/movies.dat"
ratings_path = f"file://{base_path}/ratings.dat"

movies_df = spark.read.format("csv") \
    .option("sep", "::") \
    .option("escape", "\\") \
    .schema(movies_schema) \
    .load(movies_path)

ratings_df = spark.read.format("csv") \
    .option("sep", "::") \
    .option("escape", "\\") \
    .schema(ratings_schema) \
    .load(ratings_path)

# сохраняет результат вычисления для повторного использования
ratings_df.cache()

print("Movies table:")
movies_df.show(5, truncate=False)

print("Ratings table:")
ratings_df.show(5)


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/10/18 23:56:44 WARN Utils: Your hostname, Yanas-MacBook-Pro.local, resolves to a loopback address: 127.0.0.1; using 192.168.100.16 instead (on interface en0)
26/10/18 23:56:44 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/10/18 23:56:45 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Movies table:
+--------+----------------------------------+----------------------------+
|movie_id|title                             |genres                      |
+--------+----------------------------------+----------------------------+
|1       |Toy Story (1995)                  |Animation|Children's|Comedy |
|2       |Jumanji (1995)                    |Adventure|Children's|Fantasy|
|3       |Grumpier Old Men (1995)           |Comedy|Romance              |
|4       |Waiting to Exhale (1995)          |Comedy|Drama                |
|5       |Father of the Bride Part II (1995)|Comedy                      |
+--------+----------------------------------+----------------------------+
only showing top 5 rows
Ratings table:
+-------+--------+------+---------+
|user_id|movie_id|rating|timestamp|
+-------+--------+------+---------+
|      1|    1193|     5|978300760|
|      1|     661|     3|978302109|
|      1|     914|     3|978301968|
|      1|    3408|     4|978300275|
|      1|    2355|  

In [2]:
movies_genres_df = movies_df \
    .withColumn("genre_array", split(col("genres"), "\\|")) \
    .withColumn("genre", explode(col("genre_array")))

movies_genres_df.select("movie_id", "title", "genre").show(20, truncate=False)

+--------+----------------------------------+----------+
|movie_id|title                             |genre     |
+--------+----------------------------------+----------+
|1       |Toy Story (1995)                  |Animation |
|1       |Toy Story (1995)                  |Children's|
|1       |Toy Story (1995)                  |Comedy    |
|2       |Jumanji (1995)                    |Adventure |
|2       |Jumanji (1995)                    |Children's|
|2       |Jumanji (1995)                    |Fantasy   |
|3       |Grumpier Old Men (1995)           |Comedy    |
|3       |Grumpier Old Men (1995)           |Romance   |
|4       |Waiting to Exhale (1995)          |Comedy    |
|4       |Waiting to Exhale (1995)          |Drama     |
|5       |Father of the Bride Part II (1995)|Comedy    |
|6       |Heat (1995)                       |Action    |
|6       |Heat (1995)                       |Crime     |
|6       |Heat (1995)                       |Thriller  |
|7       |Sabrina (1995)       

In [6]:
# 2
# top 10 popular genres based on user rating count
  #inner join
 #groupby + count
top_genres_by_ratings = movies_genres_df \
    .join(ratings_df, on="movie_id") \
    .groupBy("genre") \
    .count() \
    .orderBy(col("count").desc()) \
    .limit(10)

top_genres_by_ratings.show()

# top 10 users by number of ratings
top_users = ratings_df \
    .groupBy("user_id") \
    .count() \
    .select(
        "user_id",
        col("count").alias("ratings_count")
    ) \
    .orderBy(col("ratings_count").desc()) \
    .limit(10)

top_users.show()

+----------+------+
|     genre| count|
+----------+------+
|    Comedy|356580|
|     Drama|354529|
|    Action|257457|
|  Thriller|189680|
|    Sci-Fi|157294|
|   Romance|147523|
| Adventure|133953|
|     Crime| 79541|
|    Horror| 76386|
|Children's| 72186|
+----------+------+

+-------+-------------+
|user_id|ratings_count|
+-------+-------------+
|   4169|         2314|
|   1680|         1850|
|   4277|         1743|
|   1941|         1595|
|   1181|         1521|
|    889|         1518|
|   3618|         1344|
|   2063|         1323|
|   1150|         1302|
|   1015|         1286|
+-------+-------------+



In [7]:
# 3
# top 10 movies by number of ratings
top_movies = ratings_df \
    .groupBy("movie_id") \
    .count() \
    .join(movies_df, on="movie_id", how="inner") \
    .select(
        "movie_id",
        "title",
        col("count").alias("ratings_count")  
    ) \
    .orderBy(col("ratings_count").desc()) \
    .limit(10)

top_movies.show(truncate=False)

# top 10 best movies with num of ratings >= 500
top_rated_movies = ratings_df \
    .groupBy("movie_id") \
    .agg(
        count("rating").alias("ratings_count"),
        avg("rating").alias("avg_rating")
    ) \
    .filter(col("ratings_count") >= 500) \
    .join(movies_df, on="movie_id", how="inner") \
    .select(
        "movie_id",
        "title",
        "ratings_count",
        "avg_rating"
    ) \
    .orderBy(col("avg_rating").desc()) \
    .limit(10)

top_rated_movies.show(truncate=False)

+--------+-----------------------------------------------------+-------------+
|movie_id|title                                                |ratings_count|
+--------+-----------------------------------------------------+-------------+
|2858    |American Beauty (1999)                               |3428         |
|260     |Star Wars: Episode IV - A New Hope (1977)            |2991         |
|1196    |Star Wars: Episode V - The Empire Strikes Back (1980)|2990         |
|1210    |Star Wars: Episode VI - Return of the Jedi (1983)    |2883         |
|480     |Jurassic Park (1993)                                 |2672         |
|2028    |Saving Private Ryan (1998)                           |2653         |
|589     |Terminator 2: Judgment Day (1991)                    |2649         |
|2571    |Matrix, The (1999)                                   |2590         |
|1270    |Back to the Future (1985)                            |2583         |
|593     |Silence of the Lambs, The (1991)          

## JOIN виды

`JOIN` объединяет строки двух DataFrame. Обычно используется общий ключ:

```python
df1.join(df2, on="id", how="inner")
```

Если:

```text
LEFT keys:  1 2 3
RIGHT keys:   2 3 4
```

то:

| JOIN | Результирующие ключи | Смысл |
|---|---|---|
| `inner` | `2, 3` | Только строки, имеющие соответствие с обеих сторон |
| `left` | `1, 2, 3` | Все строки слева + совпавшие справа |
| `right` | `2, 3, 4` | Все строки справа + совпавшие слева |
| `outer` | `1, 2, 3, 4` | Все строки обеих таблиц |
| `cross` | все комбинации | Каждая строка первой таблицы × каждая строка второй |

### INNER JOIN

```python
df1.join(df2, on="id", how="inner")
```

Возвращает только строки, для которых ключ присутствует **в обеих таблицах**.

```text
LEFT  ∩  RIGHT
```

### LEFT JOIN

```python
df1.join(df2, on="id", how="left")
```

Возвращает **все строки левой таблицы**. Если соответствия справа нет, значения правой таблицы будут `NULL`.

### RIGHT JOIN

```python
df1.join(df2, on="id", how="right")
```

Возвращает **все строки правой таблицы**. Если соответствия слева нет, значения левой таблицы будут `NULL`.

### FULL OUTER JOIN

```python
df1.join(df2, on="id", how="outer")
```

Возвращает **все строки обеих таблиц**. При отсутствии соответствия недостающие значения заполняются `NULL`.

```text
LEFT  ∪  RIGHT
```

### CROSS JOIN

```python
df1.crossJoin(df2)
```

Создает **декартово произведение**: каждая строка первой таблицы соединяется с каждой строкой второй.

Если:

```text
df1 = N строк
df2 = M строк
```

то результат содержит:

```text
N × M строк
```

Поэтому `crossJoin` может очень быстро создать огромный DataFrame.

### Важный момент для Spark

Обычный `JOIN` часто вызывает **Shuffle**, потому что строки с одинаковыми ключами могут находиться в разных партициях. Spark приходится перераспределять данные так, чтобы одинаковые ключи оказались вместе. Поэтому `JOIN` больших таблиц может быть дорогой операцией.

In [8]:
# 4
# create temporary SQL views
movies_df.createOrReplaceTempView("movies")
ratings_df.createOrReplaceTempView("ratings")

# top 10 best movies with num of ratings >= 500
top_rated_sql = spark.sql("""
    SELECT
        m.movie_id,
        m.title,
        COUNT(r.rating) AS ratings_count,
        AVG(r.rating) AS avg_rating
    FROM ratings r
    INNER JOIN movies m
        ON r.movie_id = m.movie_id
    GROUP BY
        m.movie_id,
        m.title
    HAVING COUNT(r.rating) >= 500
    ORDER BY avg_rating DESC
    LIMIT 10
""")

top_rated_sql.show(truncate=False)

+--------+-------------------------------------------------------------------+-------------+-----------------+
|movie_id|title                                                              |ratings_count|avg_rating       |
+--------+-------------------------------------------------------------------+-------------+-----------------+
|2019    |Seven Samurai (The Magnificent Seven) (Shichinin no samurai) (1954)|628          |4.560509554140127|
|318     |Shawshank Redemption, The (1994)                                   |2227         |4.554557700942973|
|858     |Godfather, The (1972)                                              |2223         |4.524966261808367|
|745     |Close Shave, A (1995)                                              |657          |4.52054794520548 |
|50      |Usual Suspects, The (1995)                                         |1783         |4.517106001121705|
|527     |Schindler's List (1993)                                            |2304         |4.510416666666667|
|

In [9]:
# 5
from pyspark.ml.recommendation import ALS
from pyspark.ml.evaluation import RegressionEvaluator

train_df, test_df = ratings_df.randomSplit([0.8, 0.2], seed=666)

als = ALS(
    maxIter=5,
    regParam=0.01,
    userCol="user_id",
    itemCol="movie_id",
    ratingCol="rating",
    coldStartStrategy="drop"
)

model = als.fit(train_df)

predictions = model.transform(test_df)

evaluator = RegressionEvaluator(
    metricName="rmse",
    labelCol="rating",
    predictionCol="prediction"
)

rmse = evaluator.evaluate(predictions)

print("RMSE:", rmse)

RMSE: 0.9015146394303114


In [12]:
# 6
user_id = 666

user_subset = ratings_df \
    .filter(col("user_id") == user_id) \
    .select("user_id") \
    .distinct()

recommendations = model.recommendForUserSubset(user_subset, 10)

user_recommendations = recommendations \
    .select(
        "user_id",
        explode("recommendations").alias("recommendation")
    ) \
    .select(
        "user_id",
        col("recommendation.movie_id").alias("movie_id"),
        col("recommendation.rating").alias("predicted_rating")
    )

user_recommendations_with_titles = user_recommendations \
    .join(movies_df, on="movie_id", how="inner") \
    .select(
        "user_id",
        "movie_id",
        "title",
        "predicted_rating"
    ) \
    .orderBy(col("predicted_rating").desc())

user_recommendations_with_titles.show(10, truncate=False)

+-------+--------+-----------------------------------------------------------+----------------+
|user_id|movie_id|title                                                      |predicted_rating|
+-------+--------+-----------------------------------------------------------+----------------+
|666    |2342    |Hard Core Logo (1996)                                      |7.8145595       |
|666    |3338    |For All Mankind (1989)                                     |6.465068        |
|666    |702     |Faces (1968)                                               |6.2857203       |
|666    |2823    |Spiders, The (Die Spinnen, 1. Teil: Der Goldene See) (1919)|6.166215        |
|666    |3905    |Specials, The (2000)                                       |6.1437573       |
|666    |2775    |Head On (1998)                                             |6.1374598       |
|666    |632     |Land and Freedom (Tierra y libertad) (1995)                |6.09134         |
|666    |137     |Man of the Year (1995)

In [13]:
spark.stop()
